# Final evaluation on the held-out test set

Reads the results written by `uv run python -m src train` (`results/final/*.json`) and the scikit-learn baselines
(`results/baselines/*.json`). Nothing is re-trained or tuned here; the test split is **only reported**, never used to choose anything.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.experiments.report import ORDER, TITLE, load_all, top_confusions

data = load_all(ROOT / "results")

## Accuracy / macro metrics on all three splits

In [ ]:
rows = []
for m in ORDER:
    f = data[m]["final"]
    for split, s in f["splits"].items():
        rows.append({"model": TITLE[m], "split": split, "accuracy": s["accuracy"], "precision_macro": s["precision_macro"],
                     "recall_macro": s["recall_macro"], "f1_macro": s["f1_macro"], "predict_s": s["predict_time_sec"]})
df = pd.DataFrame(rows)
df.pivot(index="model", columns="split", values="accuracy")[["train", "val", "test"]].style.format("{:.4f}")

In [ ]:
df[df.split == "test"].drop(columns="split").set_index("model").style.format("{:.4f}")

## Time (CPU) and hardware

In [ ]:
f0 = data["mlp"]["final"]
print("hardware:", f0["hardware"]["cpu"], "| logical cores:", f0["hardware"]["logical_cores"], "| torch threads:", f0["hardware"]["torch_threads"])
pd.DataFrame([{"model": TITLE[m], "fit_s": data[m]["final"]["train_time_sec"],
               "predict_10k_s": data[m]["final"]["splits"]["test"]["predict_time_sec"],
               "latency_1_sample_ms": data[m]["final"]["latency_ms_single_sample"]} for m in ORDER]).set_index("model").style.format("{:.4f}")

## Ours vs scikit-learn counterpart (same hyper-parameters, no separate tuning)

In [ ]:
rows = []
for m in ORDER:
    f, sk = data[m]["final"], data[m]["sklearn"]
    if sk:
        rows.append({"model": TITLE[m], "test_acc_ours": f["splits"]["test"]["accuracy"], "test_acc_sklearn": sk["splits"]["test"]["accuracy"],
                     "fit_s_ours": f["train_time_sec"], "fit_s_sklearn": sk["train_time_sec"], "sklearn_class": sk["sklearn_class"]})
pd.DataFrame(rows).set_index("model")

## Confusion matrices and most confused digit pairs (test)

In [ ]:
display(Image(str(ROOT / "results" / "figures" / "confusion_all.png")))
for m in ORDER:
    print(TITLE[m], top_confusions(data[m]["final"]["splits"]["test"]["confusion_matrix"], 5))

In [ ]:
display(Image(str(ROOT / "results" / "figures" / "top_confusions.png"))); display(Image(str(ROOT / "results" / "figures" / "errors.png")))